# 02 — Feature Engineering

**Seção 2**:
- 7 famílias de features
- Retorno/Volatilidade, Técnicos, Volume, Microestrutura, Exógenas, Temporais, Wavelet

E **Seção 3** — Preparação dos dados:
- Divisão temporal 70/15/15
- Normalização (StandardScaler fitado no treino)
- Construção de sequências 3D (samples, timesteps, features)

In [1]:
import sys
import os
import logging
import pandas as pd
import numpy as np

sys.path.insert(0, os.path.abspath('..'))
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')

## Carregar dados processados

In [2]:
# Carregar dados limpos da etapa 01
data_clean = {}
for name in ['petr4', 'brent', 'usdbrl', 'ibov']:
    data_clean[name] = pd.read_csv(f'../data/processed/{name}_clean.csv', index_col=0, parse_dates=True)
    print(f"{name}: {data_clean[name].shape}")

target = pd.read_csv('../data/processed/target.csv', index_col=0, parse_dates=True).squeeze()
print(f"\nTarget: {len(target)} registros, positivos: {target.mean()*100:.1f}%")

petr4: (4332, 5)
brent: (4332, 5)
usdbrl: (4332, 5)
ibov: (4332, 5)

Target: 4332 registros, positivos: 52.5%


## Aplicar Feature Engineering (7 famílias)

In [3]:
from src.features.pipeline import build_feature_dataframe

df_features, target_aligned = build_feature_dataframe(data_clean, target)

print(f"\nShape final: {df_features.shape}")
print(f"Features ({df_features.shape[1]}):")
for i, col in enumerate(sorted(df_features.columns)):
    print(f"  {i+1:3d}. {col}")

2026-04-20 14:22:02,150 - src.features.pipeline - INFO - ============================================================
2026-04-20 14:22:02,150 - src.features.pipeline - INFO - INÍCIO DA FEATURE ENGINEERING
2026-04-20 14:22:02,151 - src.features.pipeline - INFO - ============================================================
2026-04-20 14:22:02,151 - src.features.pipeline - INFO - 
[1/7] Features de retorno e volatilidade...
2026-04-20 14:22:02,157 - src.features.pipeline - INFO -   Colunas: 17
2026-04-20 14:22:02,158 - src.features.pipeline - INFO - 
[2/7] Indicadores técnicos clássicos...
2026-04-20 14:22:02,150 - src.features.pipeline - INFO - INÍCIO DA FEATURE ENGINEERING
2026-04-20 14:22:02,151 - src.features.pipeline - INFO - ============================================================
2026-04-20 14:22:02,151 - src.features.pipeline - INFO - 
[1/7] Features de retorno e volatilidade...
2026-04-20 14:22:02,157 - src.features.pipeline - INFO -   Colunas: 17
2026-04-20 14:22:02,158 - sr


Shape final: (4132, 75)
Features (75):
    1. accumulation_distribution
    2. adx_14
    3. atr_14
    4. bb_lower
    5. bb_pct
    6. bb_upper
    7. beta_ibov_21d
    8. body_ratio
    9. brent_return
   10. brent_return_5d
   11. cci_20
   12. correlation_brent_5d
   13. day_of_week_cos
   14. day_of_week_sin
   15. days_since_year_start
   16. ema_ratio_21
   17. ema_ratio_50
   18. ema_ratio_9
   19. gap
   20. garman_klass_vol
   21. ibov_return
   22. ibov_return_5d
   23. ichimoku_kijun_ratio
   24. ichimoku_senkou_a_ratio
   25. ichimoku_senkou_b_ratio
   26. ichimoku_tenkan_ratio
   27. intraday_range
   28. is_month_end
   29. is_month_start
   30. is_quarter_end
   31. log_return
   32. log_return_10d
   33. log_return_21d
   34. log_return_2d
   35. log_return_5d
   36. lower_shadow
   37. macd
   38. macd_hist
   39. macd_signal
   40. mfi_14
   41. month_cos
   42. month_sin
   43. obv_norm
   44. parkinson_vol
   45. realized_vol_10d
   46. realized_vol_21d
   47. re

In [4]:
# Verificar NaN restantes
nan_count = df_features.isna().sum()
if nan_count.sum() > 0:
    print("⚠️ Features com NaN:")
    print(nan_count[nan_count > 0])
else:
    print("✅ Nenhum NaN nas features.")

# Verificar infinitos
inf_count = np.isinf(df_features.select_dtypes(include=[np.number])).sum()
if inf_count.sum() > 0:
    print("\n⚠️ Features com Inf:")
    print(inf_count[inf_count > 0])
else:
    print("✅ Nenhum Inf nas features.")

✅ Nenhum NaN nas features.
✅ Nenhum Inf nas features.


In [5]:
# Estatísticas descritivas
display(df_features.describe().T)

,count,mean,std,min,25%,50%,75%,max
log_return,4132.0,0.000389,0.028671,-0.352367,-0.013195,0.000767,0.014285,0.200671
log_return_2d,4132.0,0.000771,0.040141,-0.454700,-0.018335,0.001461,0.021231,0.245199
log_return_5d,4132.0,0.001954,0.064321,-0.696712,-0.029614,0.004639,0.037678,0.393761
log_return_10d,4132.0,0.003867,0.094078,-0.867837,-0.042800,0.008430,0.054929,0.507535
log_return_21d,4132.0,0.007863,0.139957,-0.977017,-0.061349,0.017146,0.085703,0.629691
...,...,...,...,...,...,...,...,...
wavelet_detail_4,4132.0,-0.000022,0.028319,-0.167961,-0.016283,0.000088,0.015907,0.168147
wavelet_energy_ratio,4132.0,1.594380,0.961276,0.121475,0.911715,1.398149,2.026768,11.273309
wavelet_denoised,4132.0,0.000381,0.028427,-0.344424,-0.012958,0.000554,0.014249,0.171479
wavelet_denoised_return,4132.0,-0.000012,0.031211,-0.224459,-0.016000,-0.000587,0.015487,0.270609


## Preparação dos dados para o modelo (Seção 3)

In [6]:
from src.features.pipeline import prepare_model_data

WINDOW_SIZE = 60  # Hiperparâmetro base (será otimizado na Etapa C)

sequences, scaler, metadata = prepare_model_data(
    df_features, target_aligned, window_size=WINDOW_SIZE
)

print(f"\nResumo:")
for name, (X, y) in sequences.items():
    print(f"  {name}: X={X.shape}, y={y.shape}, positivos={y.mean()*100:.1f}%")

print(f"\nTensor 3D shape: (samples={sequences['train'][0].shape[0]}, "
      f"timesteps={sequences['train'][0].shape[1]}, "
      f"features={sequences['train'][0].shape[2]})")

2026-04-20 14:22:54,093 - src.features.pipeline - INFO - ============================================================
2026-04-20 14:22:54,094 - src.features.pipeline - INFO - PREPARAÇÃO DOS DADOS PARA O MODELO
2026-04-20 14:22:54,094 - src.features.pipeline - INFO - ============================================================
2026-04-20 14:22:54,094 - src.features.pipeline - INFO - 
1. Divisão temporal...
2026-04-20 14:22:54,095 - src.features.pipeline - INFO -   train: 2892 registros (2008-06-18 a 2021-02-09) | positivos: 51.3%
2026-04-20 14:22:54,096 - src.features.pipeline - INFO -   val: 620 registros (2021-02-10 a 2023-09-18) | positivos: 55.8%
2026-04-20 14:22:54,096 - src.features.pipeline - INFO -   test: 620 registros (2023-09-19 a 2026-04-16) | positivos: 53.7%
2026-04-20 14:22:54,099 - src.features.pipeline - INFO - 
2. Normalização (StandardScaler fitado no treino)...
2026-04-20 14:22:54,094 - src.features.pipeline - INFO - PREPARAÇÃO DOS DADOS PARA O MODELO
2026-04-20 14:2


Resumo:
  train: X=(2832, 60, 75), y=(2832,), positivos=51.6%
  val: X=(560, 60, 75), y=(560,), positivos=56.2%
  test: X=(560, 60, 75), y=(560,), positivos=53.8%

Tensor 3D shape: (samples=2832, timesteps=60, features=75)


In [7]:
# Salvar artefatos
from src.utils.serialization import save_scaler, save_feature_config

save_scaler(scaler, '../models/scaler.pkl')
save_feature_config(
    feature_names=metadata['feature_names'],
    window_size=WINDOW_SIZE,
    filepath='../models/feature_config.json'
)

# Salvar features processadas e sequências
df_features.to_csv('../data/processed/features.csv')
target_aligned.to_csv('../data/processed/target_aligned.csv', header=True)

np.savez_compressed(
    '../data/processed/sequences.npz',
    X_train=sequences['train'][0], y_train=sequences['train'][1],
    X_val=sequences['val'][0], y_val=sequences['val'][1],
    X_test=sequences['test'][0], y_test=sequences['test'][1],
)

# Salvar metadata (converter datas para string)
import json
meta_save = metadata.copy()
meta_save['dates'] = {k: [str(d) for d in v] for k, v in metadata['dates'].items()}
with open('../data/processed/metadata.json', 'w') as f:
    json.dump(meta_save, f, indent=2)

print("\n✅ Etapa 2 concluída — dados prontos para modelagem.")

2026-04-20 14:23:11,177 - src.utils.serialization - INFO - Scaler salvo: ../models/scaler.pkl
2026-04-20 14:23:11,178 - src.utils.serialization - INFO - Feature config salva: ../models/feature_config.json
2026-04-20 14:23:11,178 - src.utils.serialization - INFO - Feature config salva: ../models/feature_config.json



✅ Etapa 2 concluída — dados prontos para modelagem.
